# MP5: Model Comparison & Final Recommendation

**MajsterPlus — Customer Lapse Prediction**

In this mini-project, you will:
1. Train additional models (GradientBoosting, optional VotingClassifier)
2. Compare all models on statistical metrics, business profit, and fairness
3. Assess model interpretability
4. Write a final recommendation for MajsterPlus

**CRISP-DM Phase**: Evaluation (synthesis)

---

## 0. Setup & Reproducibility

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Library version assertions
import sklearn, pandas as pd
assert sklearn.__version__.startswith(("1.4", "1.5")), (
    f"scikit-learn {sklearn.__version__} — expected 1.4.x or 1.5.x"
)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — expected 2.x"
)
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} — expected <2.0"
)
print(f"sklearn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")
print(f"Random seed: {SEED}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Load Checkpoint from MP4

In [ ]:
import hashlib, pickle
from pathlib import Path

# Colab detection
try:
    import google.colab
    DATA_DIR = Path("/content/drive/MyDrive/PUM/2. data")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/PUM/checkpoints")
except ImportError:
    DATA_DIR = Path("../2. data")
    CHECKPOINT_DIR = Path("../checkpoints")

checkpoint_file = CHECKPOINT_DIR / "mp4_checkpoint.pkl"

if checkpoint_file.exists():
    with open(checkpoint_file, "rb") as f:
        checkpoint = pickle.load(f)
    print(f"Loaded checkpoint from MP4: {list(checkpoint.keys())}")
else:
    print(f"No checkpoint found at {checkpoint_file}")
    print("You can still proceed by running MP4 first, or loading raw data.")
    checkpoint = None

In [ ]:
if checkpoint is not None:
    X_train = checkpoint["X_train"]
    X_test = checkpoint["X_test"]
    y_train = checkpoint["y_train"]
    y_test = checkpoint["y_test"]
    feature_names = checkpoint["feature_names"]
    gender_test = checkpoint.get("gender_test")
    lr_model = checkpoint.get("lr_model")
    rf_model = checkpoint.get("rf_model")
    y_prob_lr = checkpoint.get("y_prob_lr")
    y_prob_rf = checkpoint.get("y_prob_rf")
    CAMPAIGN_COST = checkpoint.get("CAMPAIGN_COST", 80)
    EXPECTED_REVENUE = checkpoint.get("EXPECTED_REVENUE")
else:
    raise FileNotFoundError(
        "No checkpoint found. Please run MP4 solution first."
    )

print(f"Loaded checkpoint from MP4")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Campaign cost: {CAMPAIGN_COST} PLN, Expected revenue: {EXPECTED_REVENUE:.2f} PLN")

## 2. Model 1: Logistic Regression (from MP3)

Already trained — compute predictions if not loaded from checkpoint.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

y_pred_lr = lr_model.predict(X_test)
if y_prob_lr is None:
    y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

print(f"LogReg ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.4f}")

## 3. Model 2: Random Forest (from MP3)

Already trained — compute predictions if not loaded from checkpoint.

In [ ]:
y_pred_rf = rf_model.predict(X_test)
if y_prob_rf is None:
    y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print(f"RF ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")

## 4. Model 3: Gradient Boosting

**Task**: Train a GradientBoosting classifier. Use `random_state=42` for reproducibility (default hyperparameters).
Compute hard predictions (`y_pred_gb`) and probability scores (`y_prob_gb`), then print the ROC-AUC.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# YOUR CODE HERE
# Train a GradientBoosting classifier. Use random_state=42 for reproducibility (default hyperparameters).
# gb = ...
# y_pred_gb = ...
# y_prob_gb = ...
# Print the ROC-AUC

## 5. Model 4: Voting Classifier (Optional)

**Task** (optional): Create a soft-voting ensemble that averages predicted probabilities from LogReg, RF, and GB.
Use `voting="soft"` and include all three previously-trained models as estimators.

In [ ]:
from sklearn.ensemble import VotingClassifier

# YOUR CODE HERE (optional but recommended)
# Create a soft-voting ensemble of the three individual models.
# voting = ...
# y_pred_vc = ...
# y_prob_vc = ...
# Print the ROC-AUC

## 6. Multi-Criteria Comparison Table

**Task**: Build a comparison DataFrame with one row per model and these columns:
Model, Accuracy, Precision, Recall, F1, ROC-AUC, Profit (PLN).

Profit should be computed at the default threshold (0.5) using the `compute_profit` function you define below.

The `compute_profit` function takes `y_true`, `y_pred`, `revenue`, and `cost` and returns:
```
TP * (revenue - cost) + FP * (-cost)
```

In [ ]:
# YOUR CODE HERE
# 1. Define compute_profit(y_true, y_pred, revenue, cost)
# 2. Build a dictionary of models: {name: (y_pred, y_prob), ...}
# 3. Loop over models and compute all metrics + profit
# 4. Create a DataFrame and display it

### Think About This

**Key Insight**: Look at which model has the highest ROC-AUC and which has the highest profit. Are they the same? If not, why not?

**TODO**: Explain in your own words why a model with better overall discrimination (AUC) might not produce the highest profit at a specific threshold.

In [ ]:
# YOUR ANSWER HERE
# Write your explanation as print() statements or comments.
# Consider: What does AUC measure across? What does profit depend on?

## 7. Business Profit Comparison

**Task**: For each model, sweep thresholds from 0.05 to 0.95 (step 0.05) and compute profit at each threshold.
Find the optimal threshold and maximum profit per model. Also report profit at the default threshold (0.5).

Plot profit vs. threshold for all models on a single chart.

In [ ]:
# YOUR CODE HERE
# thresholds = np.arange(0.05, 1.0, 0.05)
# For each model:
#   - Compute profit at threshold 0.5
#   - Sweep all thresholds, find optimal
# Print results and create a profit-vs-threshold plot

### Beyond AUC: Differentiating Near-Identical Models

When ROC-AUC values are nearly identical (e.g., 0.84–0.85 across models), AUC alone cannot guide model selection.

**TODO**: List at least 3 criteria beyond statistical performance (ROC-AUC) that should differentiate the models. For each criterion, briefly explain why it matters.

In [ ]:
# YOUR ANSWER HERE
# List at least 3 criteria beyond AUC and explain why each matters.
# Write your answer as print() statements or comments.

## 8. Fairness Analysis

**Task**: Analyze how each model performs across gender subgroups (M and K).

For each model:
1. Split the test set by `gender_test` into M and K subgroups
2. Compute recall and precision for each subgroup
3. Calculate the recall gap: `|recall_M - recall_K|`

A recall gap > 0.05 between groups warrants concern — it means the model is systematically better at identifying at-risk customers in one group.

In [ ]:
# YOUR CODE HERE
# For each model and each gender group (M, K):
#   mask = (gender_test.values == g)
#   Compute recall and precision on the masked subset
# Then compute the recall gap |recall_M - recall_K| for each model
# Print results in a structured format
# Which model has the smallest recall gap?

## 9. Interpretability Assessment

**Task**: Compare model interpretability:
- **LogReg**: Show top 10 coefficients (by absolute value) with their signs
- **RF**: Show top 10 feature importances
- **GB**: Show top 10 feature importances

Create side-by-side horizontal bar plots for all three models.

Then rank the models from most to least interpretable and explain your reasoning.

In [ ]:
# YOUR CODE HERE
# LogReg coefficients: lr_model.coef_[0] with feature_names or X_train.columns
# RF importances: rf_model.feature_importances_
# GB importances: gb.feature_importances_
#
# Print top 10 for each model
# Create side-by-side bar plots (1 row, 3 columns)
# Rank the models from most to least interpretable

## 10. Final Recommendation

**Scenario**: You're presenting to Katarzyna Nowak, VP of Marketing at MajsterPlus. She asks: *"Why should I trust this model?"* She's not technical — she needs to understand:

- **(a)** Which model you recommend and why
- **(b)** How much money it will make
- **(c)** Whether it treats all customers fairly
- **(d)** How to explain it to the board

**TODO**: Write a recommendation that addresses all four of Katarzyna's concerns. Use specific numbers from your analysis (profit, thresholds, recall gaps, interpretability). This should be a substantive, multi-paragraph business recommendation.

In [ ]:
# YOUR CODE HERE
# Write a comprehensive recommendation addressing all 4 of Katarzyna's concerns.
# Use specific numbers from your analysis.
# Write as print() statements — this is your deliverable.

## Optional: PyCaret Enrichment (Ungraded)

If you want to explore more models quickly, uncomment the PyCaret stub below.
This section is not graded.

In [ ]:
# Optional PyCaret enrichment — uncomment to run
# !pip install pycaret
# from pycaret.classification import setup, compare_models, pull
# pycaret_df = pd.concat([X_train, y_train], axis=1)
# clf = setup(pycaret_df, target=y_train.name, session_id=42)
# best = compare_models(n_select=3)
# results = pull()
# print(results)
pass  # placeholder

---
*End of MP5 — Congratulations on completing all 5 mini-projects!*